In [1]:
import sys
from pathlib import Path

sys.path.append("..")

import pandas as pd
import joblib

from src.pipeline.full_deduplication import (
    run_country_deduplication_parallel,
)

In [3]:
# ============================================================
# LOAD DATA AND MODEL
# ============================================================

df = pd.read_parquet("../data/processed/processed_dataset.parquet")

model = joblib.load(
    "../data/artifacts/random_forest_matcher.joblib"
)

print(f"Dataset rows: {len(df):,}")
print("Countries:")
print(df["country"].value_counts())

Dataset rows: 3,653,581
Countries:
country
mng    1210351
kaz     796545
kgz     789314
arm     472261
azb     297746
tjk      87364
Name: count, dtype: int64


In [4]:
FULL_DEDUP_DIR = Path("../data/full_dedup")
TEMP_DIR = Path("../data/full_dedup/tmp_chunks")

FULL_DEDUP_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

country = "tjk"

df_country = df[df["country"] == country].copy()

In [8]:
from pathlib import Path
import pandas as pd
import gc

TEMP_DIR = Path("../data/full_dedup/tmp_chunks") / "arm"
files = sorted(TEMP_DIR.glob("*.parquet"))
BATCH_SIZE = 5


def deduplicate_edges_memory_safe(edges):

    if edges.empty:
        return edges

    idx1 = edges["idx1"].to_numpy()
    idx2 = edges["idx2"].to_numpy()

    edges["left"] = pd.Series(idx1).where(
        idx1 <= idx2,
        idx2
    ).astype("int32")

    edges["right"] = pd.Series(idx2).where(
        idx1 <= idx2,
        idx1
    ).astype("int32")

    edges = edges[
        ["left", "right", "duplicate_score"]
    ]

    edges = edges.rename(
        columns={
            "left": "idx1",
            "right": "idx2"
        }
    )

    edges["duplicate_score"] = (
        edges["duplicate_score"]
        .astype("float32")
    )

    edges = edges.sort_values(
        ["idx1", "idx2", "duplicate_score"],
        ascending=[True, True, False]
    )

    edges = edges.drop_duplicates(
        subset=["idx1", "idx2"],
        keep="first"
    )

    return edges.reset_index(drop=True)
    
temp_path = Path(TEMP_DIR)

files = sorted(temp_path.glob("*.parquet"))

print(f"Found parquet chunks: {len(files)}")

parts = []

for i in range(0, len(files), BATCH_SIZE):

    batch_files = files[i:i+BATCH_SIZE]

    print(
        f"Loading batch {i//BATCH_SIZE + 1} "
        f"({len(batch_files)} files)"
    )

    batch = pd.concat(
        [pd.read_parquet(f) for f in batch_files],
        ignore_index=True
    )

    batch = deduplicate_edges_memory_safe(batch)

    parts.append(batch)

    del batch
    gc.collect()

print("Final merge...")

result_edges = pd.concat(
    parts,
    ignore_index=True
)

result_edges = deduplicate_edges_memory_safe(
    result_edges
)

print(result_edges.shape)

result_edges.to_parquet(
    "edges_arm_final.parquet",
    index=False
)

print("Saved: edges_arm_final.parquet")

Found parquet chunks: 66
Loading batch 1 (5 files)
Loading batch 2 (5 files)
Loading batch 3 (5 files)
Loading batch 4 (5 files)
Loading batch 5 (5 files)
Loading batch 6 (5 files)
Loading batch 7 (5 files)
Loading batch 8 (5 files)
Loading batch 9 (5 files)
Loading batch 10 (5 files)
Loading batch 11 (5 files)
Loading batch 12 (5 files)
Loading batch 13 (5 files)
Loading batch 14 (1 files)
Final merge...
(18730704, 3)
Saved: edges_arm_final.parquet


In [10]:
from pathlib import Path

FULL_DEDUP_DIR = Path("../data/full_dedup")

countries = ["arm", "azb", "kaz", "kgz", "mng", "tjk"]

for country in countries:
    path = FULL_DEDUP_DIR / f"edges_{country}.parquet"
    print(country, path.exists(), path)

arm True ..\data\full_dedup\edges_arm.parquet
azb False ..\data\full_dedup\edges_azb.parquet
kaz False ..\data\full_dedup\edges_kaz.parquet
kgz False ..\data\full_dedup\edges_kgz.parquet
mng False ..\data\full_dedup\edges_mng.parquet
tjk True ..\data\full_dedup\edges_tjk.parquet


In [11]:
from pathlib import Path

FULL_DEDUP_DIR = Path("../data/full_dedup")
TEMP_DIR = Path("../data/full_dedup/tmp_chunks")

countries_to_run = ["azb", "kaz", "kgz", "mng"]

for country in countries_to_run:
    print("\n" + "#" * 100)
    print(f"COUNTRY: {country}")
    print("#" * 100)

    output_path = FULL_DEDUP_DIR / f"edges_{country}.parquet"

    if output_path.exists():
        print(f"Already exists, skipping: {output_path}")
        continue

    df_country = df[df["country"] == country].copy()

    run_country_deduplication_parallel(
        df_country=df_country,
        model=model,
        country=country,
        output_path=output_path,
        temp_dir=TEMP_DIR,
        block_cols=[
            "block_prefix_len",
            "block_first_token_len",
            "block_sorted_tokens",
        ],
        max_block_size=500,
        pair_chunk_size=500_000,
        score_threshold=0.90,
        n_jobs=8,
    )


####################################################################################################
COUNTRY: azb
####################################################################################################
COUNTRY: azb
Rows: 297,746
n_jobs: 8
pair_chunk_size: 500,000
Preparing chunks for block_col=block_prefix_len


Prepare azb/block_prefix_len:   0%|          | 0/16581 [00:00<?, ?it/s]

Preparing chunks for block_col=block_first_token_len


Prepare azb/block_first_token_len:   0%|          | 0/14737 [00:00<?, ?it/s]

Preparing chunks for block_col=block_sorted_tokens


...

Prepare mng/block_sorted_tokens:   0%|          | 0/396544 [00:00<?, ?it/s]

Total chunks to score: 128


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   33.7s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:  1.1min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:  1.2min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  2.2min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:  2.9min
[Parallel(n_jobs=8)]: Done  45 tasks      | elapsed:  3.5min
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  4.1min
[Parallel(n_jobs=8)]: Done  69 tasks      | elapsed:  5.2min
[Parallel(n_jobs=8)]: Done  82 tasks      | elapsed:  6.4min
[Parallel(n_jobs=8)]: Done  97 tasks      | elapsed:  7.4min
[Parallel(n_jobs=8)]: Done 112 tasks      | elapsed:  8.2min
[Parallel(n_jobs=8)]: Done 126 out of 128 | elapsed:  9.3min remaining:    8.8s
[Parallel(n_jobs=8)]: Done 128 out of 128 | elapsed:  9.3min finished


Loading chunks mng:   0%|          | 0/128 [00:00<?, ?it/s]

Saved final country edges: ..\data\full_dedup\edges_mng.parquet
Edges found: 4,307,799


In [13]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm


class UnionFind:
    def __init__(self):
        self.parent = {}
        self.size = {}

    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.size[x] = 1

    def find(self, x):
        self.add(x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra
        self.size[ra] += self.size[rb]


def build_clusters_streaming(
    df,
    edges_paths,
    threshold=0.90,
    chunksize_rows=None,
):
    uf = UnionFind()

    # добавляем все строки как одиночные сущности
    for idx in tqdm(df.index, desc="Init nodes"):
        uf.add(int(idx))

    for path in tqdm(edges_paths, desc="Reading edge files"):
        path = Path(path)

        if not path.exists():
            continue

        edges = pd.read_parquet(path)

        if edges.empty:
            continue

        edges = edges.loc[
            edges["duplicate_score"] >= threshold,
            ["idx1", "idx2"]
        ]

        for a, b in edges.itertuples(index=False, name=None):
            uf.union(int(a), int(b))

        del edges

    roots = {}
    cluster_ids = []
    next_cluster_id = 0

    for idx in tqdm(df.index, desc="Assign clusters"):
        root = uf.find(int(idx))

        if root not in roots:
            roots[root] = next_cluster_id
            next_cluster_id += 1

        cluster_ids.append(roots[root])

    out = df.copy()
    out["cluster_id"] = cluster_ids

    return out

In [14]:
edges_paths = sorted(Path("../data/full_dedup").glob("edges_*.parquet"))

deduplicated = build_clusters_streaming(
    df=df,
    edges_paths=edges_paths,
    threshold=0.90,
)

deduplicated.to_parquet(
    "../data/full_dedup/deduplicated_records.parquet",
    index=False
)

print(deduplicated.shape)
print("clusters:", deduplicated["cluster_id"].nunique())

Init nodes:   0%|          | 0/3653581 [00:00<?, ?it/s]

Reading edge files:   0%|          | 0/6 [00:00<?, ?it/s]

Assign clusters:   0%|          | 0/3653581 [00:00<?, ?it/s]

(3653581, 18)
clusters: 1709377


In [15]:
entities = (
    deduplicated
    .sort_values(["cluster_id", "name_latin"])
    .groupby("cluster_id", as_index=False)
    .first()
)

entities.to_parquet(
    "../data/full_dedup/entities.parquet",
    index=False
)

print(entities.shape)

(1709377, 18)


In [16]:
print(deduplicated.shape)
print(entities.shape)

print("Records:", len(deduplicated))
print("Entities:", len(entities))
print("Duplicates removed:", len(deduplicated) - len(entities))

(3653581, 18)
(1709377, 18)
Records: 3653581
Entities: 1709377
Duplicates removed: 1944204


In [18]:
cluster_stats = (
    deduplicated
    .groupby("cluster_id")
    .agg(
        cluster_size=("cluster_id", "size"),
        unique_names=("name_latin", "nunique"),
        unique_countries=("country", "nunique"),
    )
    .reset_index()
)

In [19]:
cluster_stats["is_suspicious"] = (
    (cluster_stats["cluster_size"] > 20)
    | (cluster_stats["unique_names"] > 5)
    | (cluster_stats["unique_countries"] > 1)
)

cluster_stats["is_suspicious"].value_counts()

is_suspicious
False    1701239
True        8138
Name: count, dtype: int64

In [20]:
cluster_stats.sort_values(
    "cluster_size",
    ascending=False
).head(20)

,cluster_id,cluster_size,unique_names,unique_countries,is_suspicious
376991,376991,493,1,1,True
3110,3110,492,1,1,True
165,165,490,1,1,True
305,305,485,1,1,True
174,174,478,1,1,True
989,989,473,1,1,True
3286,3286,470,1,1,True
33,33,468,1,1,True
871,871,462,1,1,True
257,257,459,1,1,True


In [21]:
deduplicated = deduplicated.merge(
    cluster_stats[["cluster_id", "cluster_size", "unique_names", "unique_countries", "is_suspicious"]],
    on="cluster_id",
    how="left"
)

In [22]:
top_clusters = (
    cluster_stats
    .sort_values("cluster_size", ascending=False)
    .head(20)["cluster_id"]
    .tolist()
)

for cid in top_clusters[:5]:
    print("=" * 100)
    print("cluster_id:", cid)

    display(
        deduplicated[deduplicated["cluster_id"] == cid][
            ["country", "party_name", "name_latin", "duplicate_score"]
            if "duplicate_score" in deduplicated.columns
            else ["country", "party_name", "name_latin"]
        ]
        .drop_duplicates()
        .head(30)
    )

cluster_id: 376991


,country,party_name,name_latin
777198,kaz,ЕПАНЕШНИКОВА ИРИНА АЛЕКСАНДРОВНА,epaneshnikova irina aleksandrovna


cluster_id: 3110


,country,party_name,name_latin
5360,arm,Վաչե Մանուկյան,vache manoukyan


cluster_id: 165


,country,party_name,name_latin
171,arm,Սարգիս Կարապետյան,sargis karapetyan
36259,arm,ՍԱՐԳԻՍ ԿԱՐԱՊԵՏՅԱՆ,sargis karapetyan


cluster_id: 305


,country,party_name,name_latin
319,arm,ՎԵՐՈՆԻԿԱ ԶՈՆԱԲԵՆԴ,veronika zonabend
15338,arm,Վերոնիկա Զոնաբենդ,veronika zonabend


cluster_id: 174


,country,party_name,name_latin
180,arm,Վահե Պետրոսյան,vahe petrosyan


In [23]:
deduplicated.to_parquet("../data/full_dedup/deduplicated_records_checked.parquet", index=False)
cluster_stats.to_parquet("../data/full_dedup/cluster_stats.parquet", index=False)